## SEF Y MEF

Para cada segundo (fila) se calcula la potencia total (suma de la potencia de todas las frecuencias): P_total = 10 + 20 + 40 + 20 + 10 + ...

Se calcula la potencia acumulada: 
Frec1 = Pot1, Frec2 = Pot1 + Pot2, Frec3 = Pot1 + Pot2 + Pot3, ...


### SEF: primera frecuencia donde la acumulada alcanza el 95% de la potencia total.

Frecuencia donde la potencia acumulada alcanza el 95%. Percentil 95 de la distribución acumulada de la potencia espectral.
 - SEF = f donde ∑_i=1_f_Pi / ∑Pi ≥ 0.95


### MEF: primera frecuencia donde la acumulada alcanza el 50% de la potencia total.

Mediana de la distribución de potencia espectral.
 - MEF = f donde ∑_i=1_f_Pi / ∑Pi ≥ 0.50


Esto se debe calcular sobre potencia lineal, no sobre dB. 

pot_media = (
    df_dsa_ch1_lin[cols_freq].to_numpy(dtype=float) +
    df_dsa_ch2_lin[cols_freq].to_numpy(dtype=float)
) / 2

In [1]:
def calcular_sef_mef_desde_potencia(
    potencia,
    frecuencias,
    percentil_sef=0.95,
    percentil_mef=0.50
):
    """
    Calcula SEF y MEF a partir de una matriz de potencia lineal.

    Parámetros:
    - potencia: matriz numpy o DataFrame con forma tiempo x frecuencia.
    - frecuencias: array/lista con las frecuencias correspondientes a las columnas.
    - percentil_sef: por defecto 0.95 para SEF95.
    - percentil_mef: por defecto 0.50 para frecuencia mediana.

    Devuelve:
    - sef: array con la frecuencia bajo la cual se acumula el 95% de la potencia.
    - mef: array con la frecuencia bajo la cual se acumula el 50% de la potencia.
    """

    potencia = np.asarray(potencia, dtype=float)
    frecuencias = np.asarray(frecuencias, dtype=float)

    sef = np.full(potencia.shape[0], np.nan)
    mef = np.full(potencia.shape[0], np.nan)

    for i in range(potencia.shape[0]):
        p = potencia[i, :]

        mask = np.isfinite(p) & (p >= 0)

        if mask.sum() == 0:
            continue

        p_valid = p[mask]
        f_valid = frecuencias[mask]

        potencia_total = np.sum(p_valid)

        if potencia_total <= 0:
            continue

        acumulada = np.cumsum(p_valid)
        proporcion = acumulada / potencia_total

        idx_mef = np.searchsorted(proporcion, percentil_mef)
        idx_sef = np.searchsorted(proporcion, percentil_sef)

        mef[i] = f_valid[min(idx_mef, len(f_valid) - 1)]
        sef[i] = f_valid[min(idx_sef, len(f_valid) - 1)]

    return sef, mef